# PatchTST MSE validation v1

目标：把 `knowledge_ingest` 中 `arxiv:2211.14730` 和 `github:thuml/Time-Series-Library` 两个 PatchTST 外部知识候选补成 `patch_tst_mse` 的 research evidence。该 notebook 只验证当前 `src/model` adapter 与 backlog 证据链，不复制论文全文、不 vendoring 第三方实现代码。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact
from src.model.capabilities import model_capability_table
from src.model.sequence import PatchTSTTransformerRegressor
from src.model.training import fit_estimator_for_model, fit_predict_model, model_adapter_name, model_adapter_status

EXPERIMENT = "patch_tst_mse_validation_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_IDS = ["arxiv:2211.14730", "github:thuml/Time-Series-Library"]
ARXIV_URL = "https://arxiv.org/abs/2211.14730"
GITHUB_URL = "https://github.com/thuml/Time-Series-Library"

frame = load_soxl_support_frame(start="2026-01-01", end="2026-06-09")
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= "2026-04-30"].copy()
test = dataset.frame.loc[dataset.frame["date"] > "2026-04-30"].copy()
feature_columns = dataset.feature_columns[:5]

predictions = fit_predict_model(
    "patch_tst_mse",
    train=train,
    test=test,
    feature_columns=feature_columns,
)
estimator, constant_prediction = fit_estimator_for_model(
    "patch_tst_mse",
    train=train,
    feature_columns=feature_columns,
)
artifact = fit_model_artifact("patch_tst_mse", train=train, feature_columns=feature_columns)
artifact_metadata = artifact.metadata_payload()
capability = model_capability_table(("patch_tst_mse",)).iloc[0].to_dict()

assert constant_prediction is None
assert isinstance(estimator, PatchTSTTransformerRegressor)
assert predictions["prediction"].notna().all()
assert model_adapter_status("patch_tst_mse") == "paper_strict_adapter"
assert model_adapter_name("patch_tst_mse") == "patch_tst_transformer_mse_v1"
assert capability["paper_fidelity"] == "native_sequence"
assert capability["is_proxy"] is False
assert hasattr(estimator, "model_")
assert hasattr(estimator.model_, "encoder")
assert artifact_metadata["model_adapter"] == "patch_tst_transformer_mse_v1"

source_provenance = {
    "schema_version": "knowledge.source_provenance.v1",
    "candidate_name": "patch_tst_mse",
    "source_ids": SOURCE_IDS,
    "sources": [
        {
            "source_id": "arxiv:2211.14730",
            "source_type": "arxiv",
            "title": "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers",
            "url": ARXIV_URL,
            "summary": "PatchTST patching and channel-independence reference for the patch_tst_mse adapter backlog.",
            "retrieved_at": "2026-06-19",
            "access_status": "metadata_accessible",
            "license_note": "只保存 arXiv URL、自有摘要和验证假设；未复制论文全文。",
        },
        {
            "source_id": "github:thuml/Time-Series-Library",
            "source_type": "github",
            "title": "Time-Series-Library",
            "url": GITHUB_URL,
            "summary": "Reference implementation hub for PatchTST and related long-series models.",
            "retrieved_at": "2026-06-19",
            "access_status": "metadata_accessible",
            "license_note": "GitHub repository page reports MIT license; no third-party source code is copied or vendored.",
        },
    ],
    "source_used_as": "hypothesis_only",
    "external_source_checks": {
        "paper_url": ARXIV_URL,
        "repository_url": GITHUB_URL,
        "repository_license": "MIT",
        "copied_external_code": False,
    },
}
(CARD / "source_provenance.json").write_text(
    json.dumps(source_provenance, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(CARD / "license_review.md").write_text(
    "# License Review\n\n"
    "## 中文结论\n\n"
    f"- arXiv 来源：{ARXIV_URL}，只保存 URL、metadata 和自有摘要。\n"
    f"- GitHub 来源：{GITHUB_URL}，仓库页面复核 license 为 MIT。\n"
    "- 未复制论文全文，未保存未授权正文，未 vendoring 论文或第三方实现代码。\n"
    "- 本 notebook 只验证本仓库已有 `src.model` PatchTST adapter 的训练、预测和 artifact metadata 契约。\n"
    "- 后续如要替换为完整论文级实现，需要单独复核论文、官方仓库、依赖 license，并新增聚焦测试。\n",
    encoding="utf-8",
)

prediction_rows = predictions.loc[:, ["date", "model", "prediction", "target_return", "adapter_status", "model_adapter"]]
prediction_rows.to_csv(CARD / "predictions.csv", index=False, lineterminator="\n")
(CARD / "artifact_metadata.json").write_text(
    json.dumps(artifact_metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

adapter_evidence = {
    "schema_version": "research.patch_tst_mse.adapter_evidence.v1",
    "model": "patch_tst_mse",
    "model_adapter": model_adapter_name("patch_tst_mse"),
    "adapter_status": model_adapter_status("patch_tst_mse"),
    "paper_fidelity": capability["paper_fidelity"],
    "is_proxy": bool(capability["is_proxy"]),
    "estimator": estimator.__class__.__name__,
    "has_model": hasattr(estimator, "model_"),
    "has_transformer_encoder": hasattr(estimator.model_, "encoder"),
    "has_patch_projection": hasattr(estimator.model_, "patch_projection"),
    "patch_size": int(estimator.patch_size),
    "stride": int(estimator.stride),
    "patch_count": int(estimator.patch_count_),
    "d_model": int(estimator.d_model),
    "nhead": int(estimator.nhead),
    "num_layers": int(estimator.num_layers),
    "feature_count": estimator.feature_count_,
    "lookback": estimator.lookback_,
    "prediction_row_count": int(len(predictions)),
    "prediction_non_null": bool(predictions["prediction"].notna().all()),
    "train_row_count": int(len(train)),
    "test_row_count": int(len(test)),
    "feature_count_used": int(len(feature_columns)),
}
(CARD / "adapter_evidence.json").write_text(
    json.dumps(adapter_evidence, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

summary = {
    "schema_version": "research.patch_tst_mse_validation.v1",
    "status": "completed",
    "source_ids": SOURCE_IDS,
    "candidate_name": "patch_tst_mse",
    "target_subsystem": "src/model",
    "target_interface": "src/model.registry.ModelSpec + src.model.training.fit_predict_model",
    "model_adapter": adapter_evidence["model_adapter"],
    "adapter_status": adapter_evidence["adapter_status"],
    "paper_fidelity": adapter_evidence["paper_fidelity"],
    "is_proxy": adapter_evidence["is_proxy"],
    "estimator": adapter_evidence["estimator"],
    "prediction_row_count": adapter_evidence["prediction_row_count"],
    "prediction_non_null": adapter_evidence["prediction_non_null"],
    "implementation_allowed": False,
    "promotion_target": "research_only",
    "evidence_requirements": [
        "license_review",
        "research_notebook",
        "focused_pytest",
        "manifest_or_export",
        "chinese_conclusion",
    ],
    "artifacts": {
        "adapter_evidence": "adapter_evidence.json",
        "artifact_metadata": "artifact_metadata.json",
        "predictions": "predictions.csv",
        "source_provenance": "source_provenance.json",
        "license_review": "license_review.md",
    },
}
(CARD / "patch_tst_mse_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
summary


{'schema_version': 'research.patch_tst_mse_validation.v1',
 'status': 'completed',
 'source_ids': ['arxiv:2211.14730', 'github:thuml/Time-Series-Library'],
 'candidate_name': 'patch_tst_mse',
 'target_subsystem': 'src/model',
 'target_interface': 'src/model.registry.ModelSpec + src.model.training.fit_predict_model',
 'model_adapter': 'patch_tst_transformer_mse_v1',
 'adapter_status': 'paper_strict_adapter',
 'paper_fidelity': 'native_sequence',
 'is_proxy': False,
 'estimator': 'PatchTSTTransformerRegressor',
 'prediction_row_count': 26,
 'prediction_non_null': True,
 'implementation_allowed': False,
 'promotion_target': 'research_only',
 'evidence_requirements': ['license_review',
  'research_notebook',
  'focused_pytest',
  'manifest_or_export',
  'chinese_conclusion'],
 'artifacts': {'adapter_evidence': 'adapter_evidence.json',
  'artifact_metadata': 'artifact_metadata.json',
  'predictions': 'predictions.csv',
  'source_provenance': 'source_provenance.json',
  'license_review': 'li

In [2]:
assert summary["schema_version"] == "research.patch_tst_mse_validation.v1"
assert summary["candidate_name"] == "patch_tst_mse"
assert summary["source_ids"] == SOURCE_IDS
assert summary["model_adapter"] == "patch_tst_transformer_mse_v1"
assert summary["adapter_status"] == "paper_strict_adapter"
assert summary["paper_fidelity"] == "native_sequence"
assert summary["is_proxy"] is False
assert summary["prediction_row_count"] > 0
assert summary["prediction_non_null"] is True
assert adapter_evidence["has_transformer_encoder"] is True
assert adapter_evidence["patch_size"] == 2
assert (CARD / "source_provenance.json").exists()
assert (CARD / "license_review.md").exists()
assert (CARD / "adapter_evidence.json").exists()
assert (CARD / "artifact_metadata.json").exists()
summary


{'schema_version': 'research.patch_tst_mse_validation.v1',
 'status': 'completed',
 'source_ids': ['arxiv:2211.14730', 'github:thuml/Time-Series-Library'],
 'candidate_name': 'patch_tst_mse',
 'target_subsystem': 'src/model',
 'target_interface': 'src/model.registry.ModelSpec + src.model.training.fit_predict_model',
 'model_adapter': 'patch_tst_transformer_mse_v1',
 'adapter_status': 'paper_strict_adapter',
 'paper_fidelity': 'native_sequence',
 'is_proxy': False,
 'estimator': 'PatchTSTTransformerRegressor',
 'prediction_row_count': 26,
 'prediction_non_null': True,
 'implementation_allowed': False,
 'promotion_target': 'research_only',
 'evidence_requirements': ['license_review',
  'research_notebook',
  'focused_pytest',
  'manifest_or_export',
  'chinese_conclusion'],
 'artifacts': {'adapter_evidence': 'adapter_evidence.json',
  'artifact_metadata': 'artifact_metadata.json',
  'predictions': 'predictions.csv',
  'source_provenance': 'source_provenance.json',
  'license_review': 'li

## 中文结论

- 做了什么：把 `arxiv:2211.14730` 和 `github:thuml/Time-Series-Library` 两个 PatchTST 外部来源作为同一个 `patch_tst_mse` 候选验证，确认当前 `src.model.training.fit_predict_model` 能通过 `patch_tst_transformer_mse_v1` 训练并输出非空预测。
- 核心指标：输出 `patch_tst_mse_summary.json`、`adapter_evidence.json`、`artifact_metadata.json`、`predictions.csv`、`source_provenance.json` 和 `license_review.md`；`adapter_status=paper_strict_adapter`，`paper_fidelity=native_sequence`，`is_proxy=false`，并记录 patch size、patch count 和 Transformer encoder 证据。
- regime 差异：本 notebook 只补 knowledge promotion evidence，不重新做完整 regime/window 回测；PatchTST 的 regime 表现仍以 `research/transformer_sequence_adapter_batch_v1`、`research/mlp_mixer_native_adapter_v1`、`research/autoformer_native_adapter_v1` 和 production matrix analysis 为准。
- 是否晋升：否。本卡只把两个外部知识条目从 backlog 推进到 research evidence complete，不直接进入 production，也不声明模型可交易。
- 风险：这里使用 arXiv URL、GitHub URL、MIT license 复核和自有摘要，未复制论文全文，未 vendoring 第三方代码；当前实现是紧凑 adapter，不等同于完整论文级训练规模。
- 下一步：重新执行 `knowledge_ingest`，让 promotion audit 通过真实 research card scan 自动识别 `patch_tst_mse` 的 5 项证据，并同时覆盖 paper 与 GitHub 两个来源。